# Delta Lake Performance Optimization Guide

## Overview
This notebook provides hands-on examples and explanations for key Delta Lake optimization techniques:

1. **Partitioning** - Physical data organization by column values
2. **Z-Ordering (Clustering)** - Co-locating related data for better query performance
3. **OPTIMIZE** - File compaction to reduce small files
4. **VACUUM** - Cleaning up old file versions
5. **Liquid Clustering** - Next-generation clustering that replaces partitioning + Z-order
6. **Auto Compaction** - Automatic optimization during writes

---

**Use Case Scenario**: We'll work with a sales transaction dataset to demonstrate each technique.

In [0]:
# Create a sample sales transaction table
from datetime import datetime, timedelta
import random
import builtins
from pyspark.sql.functions import *

# S3 path for all Delta tables
VOLUME_BASE_PATH = "s3://nitya-cloutech/Account/schema2"

# Generate sample data
data = []
start_date = datetime(2023, 1, 1)
regions = ['North', 'South', 'East', 'West']
products = ['Laptop', 'Phone', 'Tablet', 'Monitor', 'Keyboard']
categories = ['Electronics', 'Accessories']

for i in range(10000):
    data.append((
        i + 1,
        start_date + timedelta(days=random.randint(0, 365)),
        random.choice(regions),
        random.choice(products),
        random.choice(categories),
        builtins.round(random.uniform(100, 5000), 2),
        random.randint(1, 10)
    ))

df = spark.createDataFrame(data, 
    ['transaction_id', 'transaction_date', 'region', 'product', 'category', 'amount', 'quantity'])

# Create Delta table in S3
base_table_path = f"{VOLUME_BASE_PATH}/sales_transactions"
df.write.format('delta').mode('overwrite').save(base_table_path)

print("✓ Sample sales data created with 10,000 transactions")
print(f"✓ Date range: {start_date.date()} to {(start_date + timedelta(days=365)).date()}")
print(f"✓ S3 Location: {base_table_path}")

In [0]:
# ===== CONFIGURATION =====
# Unity Catalog and Schema
CATALOG_NAME = "demo_catalog"
SCHEMA_NAME = "demo_schema"

# S3 path for Delta tables
VOLUME_BASE_PATH = "s3://nitya-cloutech/Account/schema2"
# ==========================================

# All table locations will be created under this base path:
table_locations = {
    'sales_transactions': f"{VOLUME_BASE_PATH}/sales_transactions",
    'sales_partitioned': f"{VOLUME_BASE_PATH}/sales_partitioned",
    'sales_liquid_clustered': f"{VOLUME_BASE_PATH}/sales_liquid_clustered",
    'sales_auto_compact': f"{VOLUME_BASE_PATH}/sales_auto_compact"
}

print("\u2713 Configuration Set")
print(f"\nCatalog: {CATALOG_NAME}")
print(f"Schema: {SCHEMA_NAME}")
print(f"S3 Base Path: {VOLUME_BASE_PATH}")
print("\nTable Locations:")
for table, location in table_locations.items():
    print(f"  - {table}: {location}")

print("\n✅ All S3 paths configured and ready to use!")

In [0]:
%sql
-- Set the current catalog and schema
USE CATALOG demo_catalog;
USE SCHEMA demo_schema;

-- Verify current context
SELECT current_catalog(), current_schema();

## 1. Partitioning

### What is Partitioning?
Partitioning physically divides your data into separate directories based on column values. Each partition is stored as a subdirectory.

### How It Works:
```
/table_root/
  ├── region=North/
  │   └── part-00001.parquet
  ├── region=South/
  │   └── part-00002.parquet
  └── region=East/
      └── part-00003.parquet
```

### Benefits:
- **Partition Pruning**: Skip reading irrelevant partitions entirely
- **Faster Queries**: When filtering by partition column
- **Data Organization**: Logical separation by business dimension

### Use Cases:
✅ **Good for:**
- Date/time columns (most common) - `WHERE transaction_date = '2023-01-01'`
- Low cardinality columns (10-1000 unique values) - region, country, category
- Columns frequently used in WHERE clauses

❌ **Avoid when:**
- High cardinality columns (user_id, transaction_id) - creates too many small files
- Columns rarely filtered
- Very small tables (< 1 GB)

### Best Practices:
- Partition by date/time for time-series data
- Keep partitions between 100 MB - 10 GB each
- Limit to 1-3 partition columns
- Cardinality should be < 1000 distinct values

In [0]:
# Create a partitioned table by region using PySpark DataFrame API
VOLUME_BASE_PATH = "s3://nitya-cloutech/Account/schema2"

# Read the source data
source_df = spark.read.format("delta").load(f"{VOLUME_BASE_PATH}/sales_transactions")

print("Creating partitioned table...")
# Drop existing table if it exists to avoid location mismatch
spark.sql("DROP TABLE IF EXISTS demo_catalog.demo_schema.sales_partitioned")

# Write as an external Delta table with partitioning to S3
table_location = f"{VOLUME_BASE_PATH}/sales_partitioned"
source_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("region") \
    .option("path", table_location) \
    .saveAsTable("demo_catalog.demo_schema.sales_partitioned")

print("✓ Partitioned table created successfully")

# Query to see partition structure
detail = spark.sql("DESCRIBE DETAIL demo_catalog.demo_schema.sales_partitioned")
display(detail)

# Query with partition pruning (only reads North partition)
result = spark.sql("""
SELECT COUNT(*) as transaction_count, SUM(amount) as total_amount
FROM demo_catalog.demo_schema.sales_partitioned
WHERE region = 'North'
""")
display(result)

In [0]:
# View partition structure using S3 directory listing
from pyspark.sql.functions import *

VOLUME_BASE_PATH = "s3://nitya-cloutech/Account/schema2"
table_location = f"{VOLUME_BASE_PATH}/sales_partitioned"

# Get table information
table_info = spark.sql("DESCRIBE DETAIL demo_catalog.demo_schema.sales_partitioned").collect()[0]

print("TABLE INFORMATION")
print("=" * 80)
print(f"Location: {table_location}")
print(f"Format: {table_info['format']}")
print(f"Partition Columns: {table_info['partitionColumns']}")
print(f"Total Files: {table_info['numFiles']}")
print(f"Size: {table_info['sizeInBytes'] / (1024 * 1024):.2f} MB")

print("\n" + "=" * 80)
print("PARTITION DIRECTORY STRUCTURE")
print("=" * 80)

# List partition directories
partitions = dbutils.fs.ls(table_location)
for partition in partitions:
    if partition.name.startswith('region='):
        print(f"\n📁 {partition.name}")
        print(f"   Path: {partition.path}")
        files = dbutils.fs.ls(partition.path)
        parquet_files = [f for f in files if f.name.endswith('.parquet')]
        print(f"   Files: {len(parquet_files)} parquet file(s)")

print("\n" + "=" * 80)
print("PARTITION DATA DISTRIBUTION")
print("=" * 80)

# Show data distribution by partition
partition_stats = spark.sql("""
    SELECT 
        region,
        COUNT(*) as record_count,
        ROUND(SUM(amount), 2) as total_amount
    FROM demo_catalog.demo_schema.sales_partitioned
    GROUP BY region
    ORDER BY region
""")

for row in partition_stats.collect():
    print(f"\n📊 region={row['region']}")
    print(f"   Records: {row['record_count']:,}")
    print(f"   Total Amount: ${row['total_amount']:,.2f}")

print("\n" + "=" * 80)
print(f"✓ Partitioned by 'region' - Data is physically separated into directories")
print(f"✓ External table with S3 storage")

In [0]:
%sql
-- Query data from each partition to see the distribution
SELECT 
    region,
    COUNT(*) as transaction_count,
    ROUND(SUM(amount), 2) as total_amount,
    ROUND(AVG(amount), 2) as avg_amount,
    MIN(transaction_date) as earliest_date,
    MAX(transaction_date) as latest_date
FROM demo_catalog.demo_schema.sales_partitioned
GROUP BY region
ORDER BY region;

## 2. Z-Ordering (Clustering)

### What is Z-Ordering?
Z-Ordering is a technique that co-locates related information in the same set of files. It uses a space-filling curve algorithm to cluster multi-dimensional data.

### How It Works:
Instead of sorting by one column, Z-Order interleaves values from multiple columns, keeping similar rows physically close together.

### Benefits:
- **Better Data Skipping**: More effective min/max statistics
- **Multi-dimensional Clustering**: Works across multiple columns simultaneously
- **Faster Queries**: When filtering on Z-ordered columns

### Use Cases:
✅ **Good for:**
- High cardinality columns (product_id, customer_id)
- Columns frequently used together in WHERE clauses
- Multi-dimensional filtering queries
- Columns used in JOIN operations

### Comparison with Partitioning:

| Feature | Partitioning | Z-Ordering |
|---------|-------------|------------|
| Cardinality | Low (10-1000) | High (1000+) |
| Physical layout | Separate directories | Within files |
| Columns supported | 1-3 | Multiple |
| Data skipping | Complete partition skip | File-level skip |
| Best for | Date/time, region | Product ID, Customer ID |

### Best Practices:
- Use on high cardinality columns
- Combine with partitioning (partition by date, Z-order by customer_id)
- Run OPTIMIZE with ZORDER regularly
- Limit to 3-4 Z-order columns for best results

In [0]:
%sql
-- IMPORTANT: Z-Ordering is applied WITH the OPTIMIZE command
-- This physically reorganizes data to co-locate related information

-- Check file count BEFORE Z-ordering
DESCRIBE DETAIL demo_catalog.demo_schema.sales_partitioned;

-- Apply Z-Ordering on product and category columns
-- This will reorganize files to cluster rows with similar product/category values together
OPTIMIZE demo_catalog.demo_schema.sales_partitioned
ZORDER BY (product, category);

-- Check file count AFTER Z-ordering (files are compacted + reorganized)
DESCRIBE DETAIL demo_catalog.demo_schema.sales_partitioned;

-- Query that benefits from Z-ordering
-- This query can skip files that don't contain 'Laptop' + 'Electronics' combination
SELECT product, category, COUNT(*) as num_transactions, AVG(amount) as avg_amount
FROM demo_catalog.demo_schema.sales_partitioned
WHERE product = 'Laptop' AND category = 'Electronics'
GROUP BY product, category;

In [0]:
# Analyze file structure after Z-ordering
import builtins

VOLUME_BASE_PATH = "s3://nitya-cloutech/Account/schema2"
table_location = f"{VOLUME_BASE_PATH}/sales_partitioned"

print("FILE STATISTICS AFTER Z-ORDERING")
print("=" * 80)

# Count total files and analyze sizes
all_files = []
partitions = dbutils.fs.ls(table_location)

for partition in partitions:
    if partition.name.startswith('region='):
        files = dbutils.fs.ls(partition.path)
        parquet_files = [f for f in files if f.name.endswith('.parquet')]
        all_files.extend(parquet_files)

total_size = builtins.sum(f.size for f in all_files)
file_sizes_mb = [f.size / (1024 * 1024) for f in all_files]

print(f"Total Files: {len(all_files)}")
print(f"Total Size: {total_size / (1024 * 1024):.2f} MB")
print(f"Average File Size: {builtins.sum(file_sizes_mb) / len(file_sizes_mb):.2f} MB")
print(f"Min File Size: {builtins.min(file_sizes_mb):.2f} MB")
print(f"Max File Size: {builtins.max(file_sizes_mb):.2f} MB")

print("\n✓ Z-Ordering reorganizes data within files for better query performance")
print("✓ Files contain co-located data based on product and category columns")
print("✓ External table with S3 storage")

## 3. OPTIMIZE (File Compaction)

### What is OPTIMIZE?
OPTIMIZE combines small files into larger ones, improving query performance by reducing file overhead.

### The Small Files Problem:
- Each write creates new files (ACID requirement)
- Streaming writes create many small files
- Too many files = slow queries (file opening overhead)

### How It Works:
```
Before OPTIMIZE:          After OPTIMIZE:
├── file1.parquet (10MB)  ├── file1.parquet (1GB)
├── file2.parquet (15MB)  └── file2.parquet (1GB)
├── file3.parquet (8MB)
├── ... (100+ files)
```

### Benefits:
- **Faster Reads**: Fewer files to open and process
- **Better Compression**: Larger files compress better
- **Improved Data Skipping**: More effective statistics
- **Reduced Metadata**: Less transaction log overhead

### Use Cases:
✅ **Run OPTIMIZE when:**
- After many small writes (streaming, frequent updates)
- Table has > 1000 small files
- Query performance degrades over time
- After DELETE/UPDATE operations (creates many small files)

### Syntax Options:
```sql
-- Basic compaction
OPTIMIZE table_name;

-- Compact specific partition
OPTIMIZE table_name WHERE date = '2023-01-01';

-- Combine with Z-ordering
OPTIMIZE table_name ZORDER BY (col1, col2);
```

### Best Practices:
- Target file size: 1 GB per file (configurable)
- Run after bulk operations
- Schedule regularly for streaming tables
- Use WHERE clause to optimize specific partitions

In [0]:
# Insert multiple small batches to create the "small files problem"
import builtins
from datetime import datetime, timedelta
import random

print("Creating small files problem by inserting multiple small batches...\n")

# Insert 5 small batches
for batch in range(5):
    small_data = []
    start_date = datetime(2024, 1, 1)
    
    for i in range(100):  # Only 100 records per batch
        small_data.append((
            10000 + (batch * 100) + i + 1,
            start_date + timedelta(days=random.randint(0, 30)),
            random.choice(['North', 'South', 'East', 'West']),
            random.choice(['Laptop', 'Phone', 'Tablet', 'Monitor', 'Keyboard']),
            random.choice(['Electronics', 'Accessories']),
            builtins.round(random.uniform(100, 5000), 2),
            random.randint(1, 10)
        ))
    
    small_df = spark.createDataFrame(small_data, 
        ['transaction_id', 'transaction_date', 'region', 'product', 'category', 'amount', 'quantity'])
    
    # Append to existing table
    VOLUME_BASE_PATH = "s3://nitya-cloutech/Account/schema2"
    small_df.write.format('delta').mode('append').save(f"{VOLUME_BASE_PATH}/sales_partitioned")
    print(f"✓ Batch {batch + 1}: Inserted 100 records")

print("\n⚠️  Small files problem created! Each insert creates new small files.")

In [0]:
# Check how many files we have now
import builtins

VOLUME_BASE_PATH = "s3://nitya-cloutech/Account/schema2"
table_location = f"{VOLUME_BASE_PATH}/sales_partitioned"

all_files = []
partitions = dbutils.fs.ls(table_location)

for partition in partitions:
    if partition.name.startswith('region='):
        files = dbutils.fs.ls(partition.path)
        parquet_files = [f for f in files if f.name.endswith('.parquet')]
        all_files.extend(parquet_files)
        
        size_mb = builtins.sum(f.size for f in parquet_files) / (1024 * 1024)
        print(f"{partition.name:20s} - {len(parquet_files):3d} files ({size_mb:6.2f} MB)")

total_size = builtins.sum(f.size for f in all_files) / (1024 * 1024)
avg_size = total_size / len(all_files) if all_files else 0

print("\n" + "=" * 60)
print(f"BEFORE OPTIMIZE:")
print(f"  Total Files: {len(all_files)}")
print(f"  Total Size: {total_size:.2f} MB")
print(f"  Avg File Size: {avg_size:.2f} MB")
print("=" * 60)
print("\n⚠️  Too many small files will slow down query performance!")

In [0]:
# Check file count before optimization
from delta.tables import DeltaTable
import builtins

VOLUME_BASE_PATH = "s3://nitya-cloutech/Account/schema2"
table_location = f"{VOLUME_BASE_PATH}/sales_partitioned"

print(f"Table location: {table_location}")

# Count files before
files_before = spark.read.format('delta').load(table_location).inputFiles()
print(f"\nFiles BEFORE OPTIMIZE: {len(files_before)}")

# Run OPTIMIZE (without Z-order, just compaction)
optimize_result = spark.sql("OPTIMIZE demo_catalog.demo_schema.sales_partitioned")
display(optimize_result)

# Count files after
files_after = spark.read.format('delta').load(table_location).inputFiles()
print(f"\nFiles AFTER OPTIMIZE: {len(files_after)}")
print(f"Files removed: {len(files_before) - len(files_after)}")
print(f"Reduction: {builtins.round((1 - len(files_after)/len(files_before)) * 100, 1)}%")

In [0]:
# Show detailed comparison of file structure
import builtins

VOLUME_BASE_PATH = "s3://nitya-cloutech/Account/schema2"
table_location = f"{VOLUME_BASE_PATH}/sales_partitioned"

print("\n" + "=" * 80)
print("AFTER OPTIMIZE - FILE STRUCTURE BY PARTITION")
print("=" * 80)

all_files_after = []
partitions = dbutils.fs.ls(table_location)

for partition in partitions:
    if partition.name.startswith('region='):
        files = dbutils.fs.ls(partition.path)
        parquet_files = [f for f in files if f.name.endswith('.parquet')]
        all_files_after.extend(parquet_files)
        
        total_partition_size = builtins.sum(f.size for f in parquet_files) / (1024 * 1024)
        avg_file_size = total_partition_size / len(parquet_files) if parquet_files else 0
        
        print(f"\n{partition.name}")
        print(f"  Files: {len(parquet_files)}")
        print(f"  Total Size: {total_partition_size:.2f} MB")
        print(f"  Avg File Size: {avg_file_size:.2f} MB")

total_size_after = builtins.sum(f.size for f in all_files_after) / (1024 * 1024)
avg_size_after = total_size_after / len(all_files_after) if all_files_after else 0

print("\n" + "=" * 80)
print(f"SUMMARY AFTER OPTIMIZE:")
print(f"  Total Files: {len(all_files_after)}")
print(f"  Total Size: {total_size_after:.2f} MB")
print(f"  Avg File Size: {avg_size_after:.2f} MB")
print("=" * 80)
print("\n✅ OPTIMIZE compacted small files into larger, more efficient files!")
print("✅ Queries will now run faster with fewer files to scan!")

## 4. VACUUM (Cleanup Old Files)

### What is VACUUM?
VACUUM permanently deletes files that are no longer referenced by the Delta table, freeing up storage space.

### Why Old Files Exist:
Delta Lake uses MVCC (Multi-Version Concurrency Control):
- Old files are kept for time travel
- Enables reading at previous versions
- Allows concurrent reads during writes

### How It Works:
```
Before VACUUM:                After VACUUM (7 days):
├── file1.parquet (v1)        ├── file3.parquet (v3) ✓
├── file2.parquet (v2)        └── file4.parquet (v3) ✓
├── file3.parquet (v3) ✓
└── file4.parquet (v3) ✓
```

### Benefits:
- **Reduced Storage Costs**: Delete unused files
- **Clean Up After OPTIMIZE**: Remove old small files
- **Compliance**: Remove old data versions

### Retention Period:
Default: **7 days** (168 hours)
- Protects against deleting files that are still in use
- Allows time travel up to retention period
- Can be configured: `VACUUM table_name RETAIN 24 HOURS`

### Use Cases:
✅ **Run VACUUM when:**
- After OPTIMIZE operations (old files no longer needed)
- Storage costs are high
- After large DELETE operations
- To enforce data retention policies

⚠️ **Warnings:**
- Cannot time travel beyond retention period after VACUUM
- Fails if readers are accessing old files
- Must disable retention check for < 7 days: `SET spark.databricks.delta.retentionDurationCheck.enabled = false`

### Syntax Options:
```sql
-- Basic vacuum (7 days retention)
VACUUM table_name;

-- Custom retention period
VACUUM table_name RETAIN 168 HOURS;

-- Dry run (see what would be deleted)
VACUUM table_name DRY RUN;
```

### Best Practices:
- Run after OPTIMIZE to reclaim space
- Use DRY RUN first to preview
- Keep default 7-day retention for safety
- Schedule vacuum regularly (weekly/monthly)

In [0]:
%sql
-- Dry run to see what would be deleted
VACUUM demo_catalog.demo_schema.sales_partitioned DRY RUN;

-- Actually vacuum with 0 hours retention (for demo purposes)
-- In production, use default 7 days or longer
-- Note: On Serverless compute, retention check configuration is handled automatically
VACUUM demo_catalog.demo_schema.sales_partitioned RETAIN 48 HOURS;

In [0]:
# Show the effect of VACUUM on storage
import builtins

VOLUME_BASE_PATH = "s3://nitya-cloutech/Account/schema2"
table_location = f"{VOLUME_BASE_PATH}/sales_partitioned"

print("=" * 80)
print("AFTER VACUUM - STORAGE CLEANUP SUMMARY")
print("=" * 80)

all_files = []
partitions = dbutils.fs.ls(table_location)

for partition in partitions:
    if partition.name.startswith('region='):
        files = dbutils.fs.ls(partition.path)
        parquet_files = [f for f in files if f.name.endswith('.parquet')]
        all_files.extend(parquet_files)

total_size = builtins.sum(f.size for f in all_files) / (1024 * 1024)

print(f"\nActive Files: {len(all_files)}")
print(f"Storage Used: {total_size:.2f} MB")
print("\n✅ Old, unreferenced files have been permanently deleted")
print("✅ Storage space reclaimed - only current version files remain")
print("⚠️  Time travel is no longer possible beyond the retention period")

print("\n" + "=" * 80)
print("VACUUM Best Practice:")
print("  - Always run VACUUM after OPTIMIZE to reclaim space")
print("  - Use DRY RUN first to preview what will be deleted")
print("  - Keep 7-day retention (default) for production safety")
print("=" * 80)

## 5. Liquid Clustering (Next-Generation)

### What is Liquid Clustering?
Liquid Clustering is a **replacement** for partitioning + Z-ordering that simplifies data layout and improves performance.

### Key Advantages Over Traditional Methods:

| Feature | Partitioning + Z-Order | Liquid Clustering |
|---------|----------------------|-------------------|
| Configuration | Complex (2 steps) | Simple (1 step) |
| Re-clustering | Manual OPTIMIZE | Incremental auto-optimize |
| Cardinality | Must choose carefully | Handles any cardinality |
| Schema evolution | Difficult to change | Easy to modify |
| Small files | Common problem | Automatically managed |

### How It Works:
- Data is automatically co-located based on clustering columns
- Incremental clustering during writes (no full table rewrites)
- Flexible schema changes (add/remove clustering columns)

### Benefits:
- **Simplicity**: Single command replaces partitioning + Z-order
- **Flexibility**: Change clustering columns without full rewrite
- **Performance**: Automatic incremental optimization
- **No Small Files**: Built-in file management

### Use Cases:
✅ **Use Liquid Clustering when:**
- Building new tables (GA for new tables)
- Queries filter on multiple columns
- Access patterns change over time
- Want automatic optimization
- High cardinality columns (customer_id, product_id)

⚠️ **Migration Note:**
- New feature (GA in DBR 13.3+)
- Cannot convert existing partitioned tables directly
- Create new table with liquid clustering and migrate data

### Syntax:
```sql
CREATE TABLE table_name
CLUSTER BY (col1, col2, col3)
AS SELECT ...;
```

### Best Practices:
- Use 3-4 clustering columns
- Choose columns used in WHERE, JOIN, GROUP BY
- Start with highest cardinality first
- Let incremental clustering handle optimization

In [0]:
%sql
-- Create a table with liquid clustering
CREATE OR REPLACE TABLE demo_catalog.demo_schema.sales_liquid_clustered
CLUSTER BY (region, product, category)
LOCATION 's3://nitya-cloutech/Account/schema2/sales_liquid_clustered'
AS SELECT * FROM delta.`s3://nitya-cloutech/Account/schema2/sales_transactions`;

-- View clustering information
DESCRIBE DETAIL demo_catalog.demo_schema.sales_liquid_clustered;

-- Query that benefits from liquid clustering
SELECT region, product, SUM(amount) as total_sales
FROM demo_catalog.demo_schema.sales_liquid_clustered
WHERE region = 'North' AND product = 'Laptop'
GROUP BY region, product;

-- Modify clustering columns (easy with liquid clustering!)
ALTER TABLE demo_catalog.demo_schema.sales_liquid_clustered CLUSTER BY (product, region);

In [0]:
# Compare Liquid Clustering structure vs Partitioning
VOLUME_BASE_PATH = "s3://nitya-cloutech/Account/schema2"
table_location = f"{VOLUME_BASE_PATH}/sales_liquid_clustered"

print("=" * 80)
print("LIQUID CLUSTERING vs PARTITIONING - DIRECTORY STRUCTURE")
print("=" * 80)

print("\n📂 LIQUID CLUSTERED TABLE:")
print(f"Location: {table_location}")
print("\nDirectory Structure:")

files = dbutils.fs.ls(table_location)
parquet_files = [f for f in files if f.name.endswith('.parquet')]

print(f"\n  📁 {table_location.split('/')[-1]}/")
for idx, f in enumerate(parquet_files[:5]):
    size_mb = f.size / (1024 * 1024)
    print(f"     ├─ {f.name} ({size_mb:.2f} MB)")
if len(parquet_files) > 5:
    print(f"     └─ ... and {len(parquet_files) - 5} more file(s)")

print("\n  ✅ NO partition directories - all files in one location")
print("  ✅ Data is CLUSTERED within files by (region, product, category)")
print("  ✅ Simpler structure, easier to manage")

print("\n" + "─" * 80)

print("\n📂 PARTITIONED TABLE (for comparison):")
partitioned_location = f"{VOLUME_BASE_PATH}/sales_partitioned"
print(f"Location: {partitioned_location}")
print("\nDirectory Structure:")

partitions = dbutils.fs.ls(partitioned_location)
partition_dirs = [p for p in partitions if p.name.startswith('region=')]

print(f"\n  📁 {partitioned_location.split('/')[-1]}/")
for p in partition_dirs[:4]:
    print(f"     ├─ {p.name}")
    files_in_partition = [f for f in dbutils.fs.ls(p.path) if f.name.endswith('.parquet')]
    print(f"     │    └─ {len(files_in_partition)} parquet file(s)")

print("\n  ⚠️  Separate directories for each partition value")
print("  ⚠️  More complex structure with nested folders")

print("\n" + "=" * 80)
print("KEY DIFFERENCE:")
print("  Partitioning  = Physical directory separation")
print("  Liquid Clustering = Logical organization within files")
print("=" * 80)

## 6. Auto Compaction (Auto Optimize)

### What is Auto Compaction?
Auto Compaction automatically runs file compaction **during write operations**, preventing small files from accumulating.

### How It Works:
When enabled, Delta Lake automatically:
1. Detects small files after a write
2. Combines them into larger files
3. Happens inline with the write operation

### Two Components:

#### 1. **Optimized Writes**
- Attempts to write larger files during the write operation
- Reduces shuffle for better performance
- Property: `delta.autoOptimize.optimizeWrite`

#### 2. **Auto Compaction**
- Runs compaction after write completes
- Combines small files created by the write
- Property: `delta.autoOptimize.autoCompact`

### Benefits:
- **No Manual OPTIMIZE**: Automatic file management
- **Better Read Performance**: Fewer small files from the start
- **Ideal for Streaming**: Prevents small file accumulation

### Trade-offs:
✅ **Pros:**
- No separate OPTIMIZE jobs needed
- Consistent read performance
- Great for streaming workloads

⚠️ **Cons:**
- Slightly slower writes (compaction overhead)
- Uses more compute during writes
- May not be suitable for ultra-high-throughput writes

### Use Cases:
✅ **Enable Auto Compaction when:**
- Streaming data ingestion
- Frequent small batch writes
- Don't want to manage OPTIMIZE jobs
- Read performance is critical

❌ **Avoid when:**
- Write latency is critical
- Very high throughput writes
- Want full control over optimization timing

### Configuration:
```sql
-- Enable at table level
ALTER TABLE table_name 
SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

-- Or set at session level
SET spark.databricks.delta.optimizeWrite.enabled = true;
SET spark.databricks.delta.autoCompact.enabled = true;
```

### Best Practices:
- Enable for streaming tables by default
- Monitor write latency after enabling
- Combine with liquid clustering for best results
- Can enable one or both features independently

In [0]:
%sql
-- Create a table with auto compaction enabled
CREATE OR REPLACE TABLE demo_catalog.demo_schema.sales_auto_compact
LOCATION 's3://nitya-cloutech/Account/schema2/sales_auto_compact'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
)
AS SELECT * FROM delta.`s3://nitya-cloutech/Account/schema2/sales_transactions`;

-- Insert more data (auto compaction will trigger)
INSERT INTO demo_catalog.demo_schema.sales_auto_compact
SELECT * FROM delta.`s3://nitya-cloutech/Account/schema2/sales_transactions` LIMIT 100;

-- View table properties
SHOW TBLPROPERTIES demo_catalog.demo_schema.sales_auto_compact;

In [0]:
# Insert multiple small batches and watch auto-compaction work
import builtins
from datetime import datetime, timedelta
import random

print("=" * 80)
print("AUTO COMPACTION DEMONSTRATION")
print("=" * 80)

VOLUME_BASE_PATH = "s3://nitya-cloutech/Account/schema2"
table_location = f"{VOLUME_BASE_PATH}/sales_auto_compact"

print("\nInserting 3 small batches (50 records each)...\n")

for batch in range(3):
    # Create small batch
    small_data = []
    start_date = datetime(2024, 2, 1)
    
    for i in range(50):
        small_data.append((
            20000 + (batch * 50) + i + 1,
            start_date + timedelta(days=random.randint(0, 30)),
            random.choice(['North', 'South', 'East', 'West']),
            random.choice(['Laptop', 'Phone', 'Tablet', 'Monitor', 'Keyboard']),
            random.choice(['Electronics', 'Accessories']),
            builtins.round(random.uniform(100, 5000), 2),
            random.randint(1, 10)
        ))
    
    small_df = spark.createDataFrame(small_data, 
        ['transaction_id', 'transaction_date', 'region', 'product', 'category', 'amount', 'quantity'])
    
    # Append to table
    small_df.write.format('delta').mode('append').save(table_location)
    
    # Count files after insert
    all_files = [f for f in dbutils.fs.ls(table_location) if f.name.endswith('.parquet')]
    total_size = builtins.sum(f.size for f in all_files) / (1024 * 1024)
    
    print(f"Batch {batch + 1}: Inserted 50 records")
    print(f"  ├─ Current file count: {len(all_files)}")
    print(f"  └─ Total size: {total_size:.2f} MB\n")

print("=" * 80)
print("✅ Auto Compaction automatically managed files during writes")
print("✅ File count stays reasonable without manual OPTIMIZE")
print("✅ Ideal for streaming and frequent write workloads")
print("=" * 80)

## Comparison Summary - When to Use What?

### Decision Matrix

| Scenario | Recommended Approach |
|----------|---------------------|
| **Time-series data with date filters** | Partition by date + Z-order by other columns |
| **High cardinality columns (customer_id, product_id)** | Liquid Clustering or Z-Order (not partition) |
| **Streaming ingestion** | Liquid Clustering + Auto Compaction |
| **New table design** | Liquid Clustering (simplest, most flexible) |
| **Existing partitioned table** | Keep partitioning, add Z-order as needed |
| **Multi-dimensional queries** | Z-Order or Liquid Clustering |
| **Small files accumulation** | Run OPTIMIZE + enable Auto Compaction |
| **High storage costs** | Run VACUUM after OPTIMIZE |
| **Frequent updates/deletes** | Regular OPTIMIZE + VACUUM schedule |

---

### Recommended Workflow

#### For New Tables:
```sql
-- Modern approach: Liquid Clustering + Auto Compaction
CREATE TABLE my_table
CLUSTER BY (col1, col2, col3)
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
)
AS SELECT ...;
```

#### For Existing Tables:
```sql
-- 1. Enable auto compaction
ALTER TABLE my_table SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

-- 2. Run optimization
OPTIMIZE my_table ZORDER BY (col1, col2);

-- 3. Clean up old files
VACUUM my_table RETAIN 168 HOURS;
```

---

### Maintenance Schedule

| Task | Frequency | When |
|------|-----------|------|
| **OPTIMIZE** | Weekly/Monthly | Manual or scheduled job |
| **VACUUM** | After OPTIMIZE | Once optimization completes |
| **Check table stats** | Weekly | Monitor file count, table size |
| **Review clustering** | Quarterly | Adjust based on query patterns |

---

### Quick Reference Commands

```sql
-- Check table details
DESCRIBE DETAIL table_name;

-- Check file count
SELECT COUNT(*) FROM (SELECT input_file_name() FROM table_name GROUP BY input_file_name());

-- Optimize
OPTIMIZE table_name;
OPTIMIZE table_name ZORDER BY (col1, col2);
OPTIMIZE table_name WHERE date = '2023-01-01';

-- Vacuum
VACUUM table_name DRY RUN;
VACUUM table_name RETAIN 168 HOURS;

-- Enable auto compaction
ALTER TABLE table_name SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);
```

In [0]:
# Comprehensive comparison of all table structures
import builtins

VOLUME_BASE_PATH = "s3://nitya-cloutech/Account/schema2"

print("=" * 100)
print("COMPLETE DIRECTORY STRUCTURE COMPARISON - ALL TABLES")
print("=" * 100)

tables = [
    ("sales_transactions", "Base Table (No optimization)"),
    ("sales_partitioned", "Partitioned + Z-Ordered"),
    ("sales_liquid_clustered", "Liquid Clustering"),
    ("sales_auto_compact", "Auto Compaction Enabled")
]

for table_name, description in tables:
    location = f"{VOLUME_BASE_PATH}/{table_name}"
    
    print(f"\n{'=' * 100}")
    print(f"📊 {table_name.upper()}")
    print(f"Description: {description}")
    print(f"Location: {location}")
    print("-" * 100)
    
    try:
        # Check if location exists
        items = dbutils.fs.ls(location)
        
        # Separate directories and files
        directories = [item for item in items if item.name.endswith('/')]
        files = [item for item in items if item.name.endswith('.parquet')]
        
        if directories:
            print(f"\n  📁 Directory Structure (Partitioned):")
            for dir_item in directories[:5]:
                print(f"     ├─ {dir_item.name}")
                try:
                    dir_files = [f for f in dbutils.fs.ls(dir_item.path) if f.name.endswith('.parquet')]
                    dir_size = builtins.sum(f.size for f in dir_files) / (1024 * 1024)
                    print(f"     │    └─ {len(dir_files)} file(s) ({dir_size:.2f} MB)")
                except:
                    pass
            if len(directories) > 5:
                print(f"     └─ ... and {len(directories) - 5} more partition(s)")
        
        if files:
            print(f"\n  📝 File Structure (Non-partitioned):")
            for idx, file in enumerate(files[:5]):
                size_mb = file.size / (1024 * 1024)
                prefix = "├─" if idx < min(4, len(files) - 1) else "└─"
                print(f"     {prefix} {file.name} ({size_mb:.2f} MB)")
            if len(files) > 5:
                print(f"     └─ ... and {len(files) - 5} more file(s)")
        
        # Calculate total statistics
        all_files = []
        if directories:
            for dir_item in directories:
                try:
                    dir_files = [f for f in dbutils.fs.ls(dir_item.path) if f.name.endswith('.parquet')]
                    all_files.extend(dir_files)
                except:
                    pass
        all_files.extend(files)
        
        if all_files:
            total_size = sum(f.size for f in all_files) / (1024 * 1024)
            avg_size = total_size / len(all_files)
            
            print(f"\n  📊 Statistics:")
            print(f"     Total Files: {len(all_files)}")
            print(f"     Total Size: {total_size:.2f} MB")
            print(f"     Avg File Size: {avg_size:.2f} MB")
            print(f"     Partitions: {len(directories) if directories else 0}")
        else:
            print("\n  ⚠️  No data files found")
            
    except Exception as e:
        print(f"\n  ⚠️  Table not yet created or error accessing: {str(e)}")

print("\n" + "=" * 100)
print("SUMMARY:")
print("  • Partitioning: Physical directory separation (good for low cardinality)")
print("  • Z-Ordering: File-level data clustering (good for high cardinality)")
print("  • Liquid Clustering: Next-gen approach combining benefits of both")
print("  • Auto Compaction: Automatic file management during writes")
print("=" * 100)

## Best Practices and Key Takeaways

### 💡 Golden Rules

1. **Start with Liquid Clustering for new tables**
   - Simplest approach with best flexibility
   - Handles any cardinality automatically
   - Easy to modify clustering strategy

2. **Partition by time for time-series data**
   - Most common and effective pattern
   - Enables efficient partition pruning
   - Combine with Z-ordering for multi-dimensional queries

3. **Never partition by high cardinality columns**
   - Avoid: user_id, transaction_id, customer_id
   - Use Z-order or Liquid Clustering instead
   - High cardinality = too many small files

4. **Run OPTIMIZE regularly**
   - After bulk operations (DELETE, UPDATE, MERGE)
   - When query performance degrades
   - Or enable Auto Compaction for automatic management

5. **Always VACUUM after OPTIMIZE**
   - Reclaim storage from old files
   - Use DRY RUN first to preview
   - Keep 7-day retention for safety

---

### ⚠️ Common Pitfalls to Avoid

| Mistake | Why It's Bad | Better Approach |
|---------|-------------|----------------|
| Over-partitioning | Creates too many small files | Limit to 1-3 partition columns with < 1000 values |
| Never running OPTIMIZE | Query performance degrades | Schedule weekly/monthly OPTIMIZE |
| VACUUM too aggressively | Breaks time travel, concurrent reads | Keep 7-day retention minimum |
| Not using Auto Compaction for streaming | Small files accumulate quickly | Enable Auto Compaction for streaming tables |
| Mixing high and low cardinality in partitions | Unbalanced partition sizes | Use high cardinality for Z-order/clustering only |

---

### 🚀 Performance Optimization Checklist

**Before Designing Your Table:**
- [ ] Identify your most common query patterns
- [ ] Determine cardinality of candidate columns
- [ ] Choose partitioning strategy (time-based most common)
- [ ] Select clustering columns (high cardinality)

**For Existing Tables:**
- [ ] Check file count and sizes: `DESCRIBE DETAIL table_name`
- [ ] Run OPTIMIZE if > 1000 small files
- [ ] Apply Z-ordering on frequently filtered columns
- [ ] Run VACUUM to reclaim storage
- [ ] Enable Auto Compaction for future writes

**Monitoring:**
- [ ] Track query performance metrics
- [ ] Monitor file count over time
- [ ] Review storage costs
- [ ] Adjust clustering based on query patterns

---

### 🎯 Real-World Example Patterns

#### E-commerce Transactions Table
```sql
CREATE TABLE transactions
CLUSTER BY (customer_id, product_id, order_date)
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
)
```
**Why?** High cardinality clustering + auto file management

#### IoT Sensor Data (Time-Series)
```sql
CREATE TABLE sensor_data
PARTITIONED BY (date)
TBLPROPERTIES (
  'delta.autoOptimize.autoCompact' = 'true'
)
```
**Then:** `OPTIMIZE sensor_data ZORDER BY (sensor_id, location)`
**Why?** Time-based partition pruning + sensor clustering

#### User Activity Logs (High Volume)
```sql
CREATE TABLE user_logs
CLUSTER BY (user_id, event_type, timestamp)
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
)
```
**Why?** Liquid clustering handles high write volume + auto optimization

---

### 📚 Further Learning Resources

* **Databricks Documentation**: Delta Lake Best Practices
* **Performance Tuning**: Query optimization guides
* **Delta Lake Deep Dive**: Understanding transaction log
* **Liquid Clustering**: Migration guide and patterns

---

## Summary

You now have a complete understanding of:

✓ **Partitioning** - Physical data organization for query pruning
✓ **Z-Ordering** - Multi-dimensional clustering for data skipping
✓ **OPTIMIZE** - File compaction to reduce small files
✓ **VACUUM** - Storage cleanup and cost management
✓ **Liquid Clustering** - Next-gen approach combining best of both
✓ **Auto Compaction** - Automatic optimization during writes

**Next Steps**: Run the examples above on your own data and measure the performance improvements!